# 04 — Statistical XAI and Bayesian uncertainty diagnostics


**Purpose.** Load fresh BEDOL checkpoints created by Notebook 01. Part A estimates grouped model reliance by block-permuting complete historical input trajectories across forecast origins. Part B performs Monte Carlo latent sampling for uncertainty diagnostics. These are regenerated analyses, not cached paper results.


In [ ]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT/"src"))
from bedol_repro import *

DATA_DIR, CONFIG_DIR = ROOT/"data", ROOT/"configs"
OUT, CKPT = ROOT/"reproduction_outputs", ROOT/"reproduction_outputs"/"checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

manifest, frames = load_public_data(DATA_DIR, CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")
proc, meta, site_to_idx, hist_features, future_features = prepare_frames(frames,splits)
test_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"test")

ckpts = sorted(CKPT.glob("bedol_seed*.pt"))
if not ckpts:
    raise FileNotFoundError("Run Notebook 01 first.")
models=[]
for p in ckpts:
    obj=torch.load(p,map_location="cpu")
    m=BEDOL(len(hist_features),len(future_features),len(site_to_idx),"full").to(DEVICE)
    m.load_state_dict(obj["state_dict"]); m.eval(); models.append(m)
print("loaded checkpoints:", [p.name for p in ckpts])


In [ ]:
# A. Grouped blocked-trajectory permutation XAI.
# The target trajectory is never permuted. Only selected historical input channels
# are replaced by trajectories from permuted forecast origins within the same site.
GROUPS = {
    "Historical power":["y_norm"],
    "Temperature":[c for c in ["temp_mean","temp_min","temp_max","air_temp","ground_temp","dew_point"] if c in hist_features],
    "Moisture / precipitation / cloud":[c for c in ["precip","vapor_pressure","cloud_cover","relative_humidity"] if c in hist_features],
    "Radiation / sunshine":[c for c in ["solar_radiation","sunshine_duration"] if c in hist_features],
    "Wind / pressure":[c for c in ["wind_speed","air_pressure"] if c in hist_features],
    "Historical calendar":[c for c in ["hour_sin","hour_cos","doy_sin","doy_cos","dow_sin","dow_cos","weekend"] if c in hist_features],
}
feature_index={c:i for i,c in enumerate(hist_features)}

class PermutedHistoryDataset(Dataset):
    def __init__(self, base, columns, block=28, seed=20260929):
        self.base=base
        self.cols=[feature_index[c] for c in columns]
        self.source=np.arange(len(base))
        rng=np.random.default_rng(seed)
        # Permute blocks separately within each site so site identity is preserved.
        for site in proc:
            idx=np.array([i for i,(s,_) in enumerate(base.items) if s==site])
            blocks=[idx[j:j+block] for j in range(0,len(idx),block)]
            perm=rng.permutation(len(blocks))
            src=np.concatenate([blocks[j] for j in perm])[:len(idx)]
            self.source[idx]=src
    def __len__(self): return len(self.base)
    def __getitem__(self,k):
        hist,fut,site,y,name,origin,times=self.base[k]
        shist,*_=self.base[int(self.source[k])]
        hist=hist.clone()
        hist[:,self.cols]=shist[:,self.cols]
        return hist,fut,site,y,name,origin,times

baseline = pd.read_csv(OUT/"01_bedol_test_predictions.csv", parse_dates=["origin","target_time"])
base_loss=float(origin_loss(baseline).loss.mean())
xai=[]
for group, cols in GROUPS.items():
    if not cols: continue
    perm_ds=PermutedHistoryDataset(test_ds,cols)
    q=predict_ensemble(models,perm_ds,proc,meta,DEVICE)
    delta=float(origin_loss(q).loss.mean()-base_loss)
    xai.append({"group":group,"delta_normalized_MSE":delta})
xai=pd.DataFrame(xai).sort_values("delta_normalized_MSE",ascending=False)
xai.to_csv(OUT/"04_xai_group_reliance.csv",index=False)
display(xai)
print("These values quantify model reliance, not causal effects.")


In [ ]:
# B. Monte Carlo latent-sampling uncertainty diagnostic.
# Validation-only interval calibration is intentionally kept simple and transparent.
def mc_table(ds, draws=50):
    rows=[]
    loader=make_loader(ds,64,False)
    for hist,fut,site,y,site_names,origins,target_times in loader:
        hist,fut,site_t=hist.to(DEVICE),fut.to(DEVICE),site.to(DEVICE)
        all_draws=[]
        with torch.no_grad():
            for model in models:
                model.eval()
                for _ in range(draws):
                    p,*_=model(hist,fut,site_t,sample=True)
                    all_draws.append(p.cpu().numpy())
        arr=np.stack(all_draws,0)
        mean=arr.mean(0); lo=np.quantile(arr,.05,axis=0); hi=np.quantile(arr,.95,axis=0)
        yy=y.numpy()
        for b,name in enumerate(site_names):
            cap=meta[name]["capacity"]; raw=proc[name].set_index("ds")
            origin=pd.Timestamp(origins[b])
            for h in range(HORIZON):
                tt=origin+pd.Timedelta(hours=h)
                flag=int(raw.loc[tt,"recommended_for_evaluation"]) if tt in raw.index else 0
                rows.append((name,origin,h+1,yy[b,h]*cap,mean[b,h]*cap,lo[b,h]*cap,hi[b,h]*cap,flag))
    return pd.DataFrame(rows,columns=["site_id","origin","horizon","y_true","mean","lo90","hi90","eval_flag"])

MC_DRAWS = 20  # increase to 100+ for a more stable diagnostic
unc = mc_table(test_ds,MC_DRAWS)
u=unc[unc.eval_flag.eq(1)].copy()
u["covered"]=(u.y_true>=u.lo90)&(u.y_true<=u.hi90)
u["width"]=u.hi90-u.lo90
site_unc=u.groupby("site_id").agg(
    empirical_coverage=("covered","mean"),
    mean_interval_width=("width","mean")
).reset_index()
site_unc.to_csv(OUT/"04_uncertainty_by_site.csv",index=False)
display(site_unc)
print("Raw public MC intervals are a reproducibility diagnostic; the paper reports validation-calibrated uncertainty separately.")
